# YOLO Training Workflow

This notebook is a clean template for training, validating, and testing detection models.
It is organized to minimize repeated code and make experiments easier to track.


## 1. Environment Setup

Run this cell first to load dependencies, detect the project root, and verify CUDA.


In [22]:
from __future__ import annotations

import gc
from datetime import date
from pathlib import Path

import torch
import ultralytics
from ultralytics import YOLO
from IPython.display import Image, display

# Resolve project root (works if notebook is run from project root or notebooks/)
CWD = Path.cwd().resolve()
PROJECT_ROOT = CWD.parent if CWD.name == "notebooks" else CWD

print(f"Project root: {PROJECT_ROOT}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Torch version: {torch.__version__}")

RUN_ULTRALYTICS_CHECKS = False  # Set True only when you want a full environment check
if RUN_ULTRALYTICS_CHECKS:
    ultralytics.checks()


Project root: /home/robiotec/Documents/Entrenamientos/Training
CUDA available: True
GPU: NVIDIA GeForce RTX 5090
Torch version: 2.10.0+cu128


## 2. Shared Paths and Defaults

Keep all reusable paths and defaults in one place.


In [23]:
BASE_MODEL_PATH = PROJECT_ROOT / "model" / "yolo26n.pt"
RESULT_ROOT = PROJECT_ROOT / "result" 

YAML_CONFIGS = {
    "caja": PROJECT_ROOT / "configs" / "caja.yaml",
    "vetas": PROJECT_ROOT / "configs" / "vetas.yaml",
    "mixed": PROJECT_ROOT / "configs" / "mixto.yaml",
}

# Shared defaults for training. Override per experiment only when needed.
DEFAULT_TRAIN_ARGS = {
    "epochs": 150,
    "imgsz": 640,
    "patience": 20,
    "device": 0,
    "warmup_epochs": 3,
    "seed": 42,
    "lrf": 0.1,
    "weight_decay": 0.0001,
    "workers": 8,
    "cache": "disk",
    "plots": True,
}

DEFAULT_SPLIT_TAGS = {
    "caja": "BONANZA_SPLIT_V2",
    "vetas": "2026-04-22_TenguelVeta",
    "mixed": "2026-04-22_TenguelMixto",
}

print(f"Base model: {BASE_MODEL_PATH}")
for name, cfg in YAML_CONFIGS.items():
    print(f"{name:>5}: {cfg}")


Base model: /home/robiotec/Documents/Entrenamientos/Training/model/yolo26n.pt
 caja: /home/robiotec/Documents/Entrenamientos/Training/configs/caja.yaml
vetas: /home/robiotec/Documents/Entrenamientos/Training/configs/vetas.yaml
mixed: /home/robiotec/Documents/Entrenamientos/Training/configs/mixto.yaml


## 3. Utility Functions

These helpers remove duplicated code for CUDA cleanup, train, validate, and predict.


In [24]:
def clean_cuda(verbose: bool = True) -> None:
    # Release cached GPU memory after heavy operations.
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        if verbose:
            allocated = torch.cuda.memory_allocated() / 1024**2
            reserved = torch.cuda.memory_reserved() / 1024**2
            print(f"GPU memory - allocated: {allocated:.2f} MB | reserved: {reserved:.2f} MB")


def train_model(
    run_group: str,
    run_name: str,
    data_yaml: Path,
    model_path: Path = BASE_MODEL_PATH,
    **overrides,
):
    # Train a model and store outputs under result/<run_group>/<run_name>.
    args = dict(DEFAULT_TRAIN_ARGS)
    args.update(overrides)

    model = YOLO(str(model_path))
    output = model.train(
        data=str(data_yaml),
        project=str(RESULT_ROOT / run_group),
        name=run_name,
        **args,
    )
    clean_cuda()
    return output


def validate_model(
    model_path: Path,
    data_yaml: Path,
    run_group: str,
    run_name: str,
    split: str = "test",
    conf: float = 0.25,
    **overrides,
):
    # Run validation and save metrics under result/<run_group>/<run_name>.
    model = YOLO(str(model_path))
    output = model.val(
        data=str(data_yaml),
        split=split,
        conf=conf,
        project=str(RESULT_ROOT / run_group),
        name=run_name,
        save_json=True,
        plots=True,
        **overrides,
    )
    clean_cuda()
    return output


def predict_images(
    model_path: Path,
    source_path: Path,
    run_group: str,
    run_name: str,
    conf: float = 0.25,
    **overrides,
):
    # Run inference on images and save visual predictions.
    model = YOLO(str(model_path))
    output = model.predict(
        source=str(source_path),
        conf=conf,
        project=str(RESULT_ROOT / run_group),
        name=run_name,
        save=True,
        **overrides,
    )
    clean_cuda()
    return output


## 4. Quick GPU Cleanup

Run this anytime you want to free cached CUDA memory with one click.


In [25]:
clean_cuda()

GPU memory - allocated: 64.00 MB | reserved: 64.00 MB


## 5. Configure One Training Run

Edit only this cell for each experiment.


In [26]:
today = date.today().isoformat()

RUN_CONFIG = {
    "target": "caja",  # Use "caja", "vetas", or "mixed"
    "yaml": "caja",  # Use "caja", "vetas", or "mixed"
    "split_tag": "M17_CAJA_V1_NO_AUG",  # Name tag for this experiment/output folder.
    "dataset_split": "SPLIT_M17_V1",# Real dataset split folder used by configs/caja.yaml.
    "run_id": f"run_{today}",
    "train_args": {
        "batch": 64,  # Images per batch. Lower this if GPU memory is not enough.
        "patience": 20,  # Stop training if validation does not improve after this many epochs.
        "lr0": 0.01,  # Initial learning rate.

        # Data augmentation for this run.
        "hsv_h": 0.00,
        "hsv_s": 0.0,
        "hsv_v": 0.0,
        "degrees": 0,
        "translate": 0.0,
        "scale": 0.0,
        "shear": 0.0,
        "perspective": 0.0,
        "flipud": 0.0,
        "fliplr": 0.5,
        "mosaic": 0.0,
        "mixup": 0.0,
        "copy_paste": 0.0,
        "cutmix": 0.0,
        "erasing": 0.0,
        "bgr": 0.0,
        "multi_scale": False,
    },
}

RUN_NAME = f"{RUN_CONFIG['split_tag']}__{RUN_CONFIG['run_id']}"
DATA_YAML = YAML_CONFIGS[RUN_CONFIG["yaml"]]
RUN_GROUP = RUN_CONFIG["target"]
RESULT_DIR = RESULT_ROOT / RUN_GROUP / RUN_NAME
MODEL_TO_VALIDATE = RESULT_DIR / "weights" / "best.pt"

if not DATA_YAML.exists():
    raise FileNotFoundError(f"No encuentro el data.yaml del split: {DATA_YAML}")

print(f"Training group: {RUN_GROUP}")
print(f"Run name: {RUN_NAME}")
print(f"YAML config: {DATA_YAML}")
print(f"Dataset split: {RUN_CONFIG['dataset_split']}")
print(f"Expected result dir: {RESULT_DIR}")


Training group: caja
Run name: M17_CAJA_V1_NO_AUG__run_2026-06-03
YAML config: /home/robiotec/Documents/Entrenamientos/Training/configs/caja.yaml
Dataset split: SPLIT_M17_V1
Expected result dir: /home/robiotec/Documents/Entrenamientos/Training/result/caja/M17_CAJA_V1_NO_AUG__run_2026-06-03


## 6. Train


In [27]:
train_results = train_model(
    run_group=RUN_GROUP,
    run_name=RUN_NAME,
    data_yaml=DATA_YAML,
    **RUN_CONFIG["train_args"],
)



New https://pypi.org/project/ultralytics/8.4.60 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.13 🚀 Python-3.10.19 torch-2.10.0+cu128 CUDA:0 (NVIDIA GeForce RTX 5090, 32103MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=64, bgr=0.0, box=7.5, cache=disk, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/robiotec/Documents/Entrenamientos/Training/configs/caja.yaml, degrees=0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=150, erasing=0.0, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.0, hsv_s=0.0, hsv_v=0.0, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.1, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/home/robiotec/Documents/Entren

## 7. Validate on Test Split

Set `MODEL_TO_VALIDATE` to your best checkpoint.


In [ ]:
if not MODEL_TO_VALIDATE.exists():
    raise FileNotFoundError(
        f"No encuentro el modelo entrenado en {MODEL_TO_VALIDATE}. Ejecuta primero la celda de entrenamiento."
    )

val_results = validate_model(
    model_path=MODEL_TO_VALIDATE,
    data_yaml=DATA_YAML,
    run_group=RUN_GROUP,
    run_name=f"{RUN_NAME}__val_test",
    split="test",
    conf=0.70,
)


## 8. Predict on External Images (Optional)

Set `PREDICT_SOURCE` to any folder of images.


In [ ]:
PREDICT_SOURCE = PROJECT_ROOT / "data" / "splits" / RUN_CONFIG["dataset_split"] / "test" / "images"

# Uncomment to run prediction on the test images from this split
# pred_results = predict_images(
#     model_path=MODEL_TO_VALIDATE,
#     source_path=PREDICT_SOURCE,
#     run_group=RUN_GROUP,
#     run_name=f"{RUN_NAME}__predict",
#     conf=0.25,
# )


## 9. Quick Visual Comparison (Optional)


In [ ]:
# Update image paths if you want to compare confusion matrices side by side.
# from IPython.display import Image, display
#
# img1 = Image(filename=str(RESULT_ROOT / "Caja" / "some_run" / "confusion_matrix_normalized.png"), width=450)
# img2 = Image(filename=str(RESULT_ROOT / "Caja" / "some_run__val_test" / "confusion_matrix_normalized.png"), width=450)
#
# try:
#     from ipywidgets import HBox
#     display(HBox([img1, img2]))
# except Exception:
#     # Fallback without ipywidgets
#     display(img1)
#     display(img2)
